# Buy-and-Hold Strategy

In [1]:
import numpy as np
import pandas as pd

## 1. Performance Metric Functions

In [2]:
def compute_cumulative_return(returns: pd.Series) -> float:
    """
    Compute total cumulative return from a series of simple returns.
    """
    returns = returns.dropna()
    if len(returns) == 0:
        return np.nan
    return (1 + returns).prod() - 1


def compute_annualized_return(returns: pd.Series, periods_per_year: int = 8760) -> float:
    """
    Compute annualized return from a series of simple hourly returns.
    """
    returns = returns.dropna()
    if len(returns) == 0:
        return np.nan

    cumulative = (1 + returns).prod()
    n_periods = len(returns)

    if cumulative <= 0:
        return np.nan

    return cumulative ** (periods_per_year / n_periods) - 1


def compute_annualized_volatility(returns: pd.Series, periods_per_year: int = 8760) -> float:
    """
    Compute annualized volatility from hourly returns.
    """
    returns = returns.dropna()
    if len(returns) < 2:
        return np.nan
    return returns.std(ddof=1) * np.sqrt(periods_per_year)


def compute_annualized_sharpe(
    returns: pd.Series,
    risk_free_rate: float = 0.0,
    periods_per_year: int = 8760
) -> float:
    """
    Compute annualized Sharpe ratio using hourly returns.

    Assumes risk_free_rate is annualized (e.g. 0.02 for 2% yearly).
    """
    returns = returns.dropna()
    if len(returns) < 2:
        return np.nan

    rf_per_period = risk_free_rate / periods_per_year
    excess_returns = returns - rf_per_period

    vol = excess_returns.std(ddof=1)
    if vol == 0 or np.isnan(vol):
        return np.nan

    sharpe = excess_returns.mean() / vol
    return sharpe * np.sqrt(periods_per_year)


def compute_max_drawdown(returns: pd.Series) -> float:
    """
    Compute maximum drawdown from a series of simple returns.
    """
    returns = returns.dropna()
    if len(returns) == 0:
        return np.nan

    equity_curve = (1 + returns).cumprod()
    running_max = equity_curve.cummax()
    drawdown = equity_curve / running_max - 1

    return drawdown.min()


def compute_turnover(position: pd.Series) -> float:
    """
    Compute turnover as the average absolute change in position.

    For buy-and-hold:
    - initial entry is counted separately via num_trades
    - thereafter position stays constant
    """
    position = position.fillna(0)
    position_change = position.diff().abs().fillna(position.iloc[0])
    return position_change.sum()


def compute_num_trades(position: pd.Series) -> int:
    """
    Count number of position changes.
    For buy-and-hold:
    - entering from 0 to 1 at the start counts as 1 trade
    """
    position = position.fillna(0)
    position_change = position.diff().abs().fillna(position.iloc[0])
    return int((position_change > 0).sum())


def compute_trade_returns(net_returns: pd.Series, position: pd.Series) -> list:
    """
    Compute returns per trade by grouping consecutive invested periods.

    For buy-and-hold, there is typically only one trade spanning the full test sample.
    """
    net_returns = net_returns.reset_index(drop=True)
    position = position.reset_index(drop=True)

    trade_returns = []
    in_trade = False
    current_trade_returns = []

    for r, p in zip(net_returns, position):
        if p != 0 and not in_trade:
            in_trade = True
            current_trade_returns = [r]
        elif p != 0 and in_trade:
            current_trade_returns.append(r)
        elif p == 0 and in_trade:
            trade_returns.append((1 + pd.Series(current_trade_returns)).prod() - 1)
            in_trade = False
            current_trade_returns = []

    if in_trade and len(current_trade_returns) > 0:
        trade_returns.append((1 + pd.Series(current_trade_returns)).prod() - 1)

    return trade_returns


def summarize_performance(
    backtest_df: pd.DataFrame,
    return_col: str = "net_return",
    position_col: str = "position",
    periods_per_year: int = 8760,
    risk_free_rate: float = 0.0
) -> pd.Series:
    """
    Generate a full performance summary for the strategy.
    """
    returns = backtest_df[return_col].dropna()
    position = backtest_df[position_col]

    cumulative_return = compute_cumulative_return(returns)
    annualized_return = compute_annualized_return(returns, periods_per_year)
    annualized_volatility = compute_annualized_volatility(returns, periods_per_year)
    annualized_sharpe = compute_annualized_sharpe(returns, risk_free_rate, periods_per_year)
    max_drawdown = compute_max_drawdown(returns)
    turnover = compute_turnover(position)
    num_trades = compute_num_trades(position)

    trade_returns = compute_trade_returns(backtest_df[return_col], backtest_df[position_col])
    avg_return_per_trade = np.mean(trade_returns) if len(trade_returns) > 0 else np.nan

    summary = pd.Series({
        "Cumulative Return": cumulative_return,
        "Annualized Return": annualized_return,
        "Annualized Volatility": annualized_volatility,
        "Annualized Sharpe": annualized_sharpe,
        "Maximum Drawdown": max_drawdown,
        "Turnover": turnover,
        "Number of Trades": num_trades,
        "Average Return per Trade": avg_return_per_trade
    })

    return summary

## 2. Classification Metric Functions

In [4]:
def compute_directional_metrics(backtest_df: pd.DataFrame) -> pd.Series:
    """
    Optional directional metrics for comparison.
    Since buy-and-hold predicts 'up' every time, this is not a true classifier,
    but the metrics can still be computed for consistency.

    Predicted positive if position == 1
    Actual positive if raw_return > 0
    """
    df = backtest_df.copy()

    df["predicted_positive"] = (df["position"] == 1).astype(int)
    df["actual_positive"] = (df["raw_return"] > 0).astype(int)

    tp = ((df["predicted_positive"] == 1) & (df["actual_positive"] == 1)).sum()
    fp = ((df["predicted_positive"] == 1) & (df["actual_positive"] == 0)).sum()
    fn = ((df["predicted_positive"] == 0) & (df["actual_positive"] == 1)).sum()
    tn = ((df["predicted_positive"] == 0) & (df["actual_positive"] == 0)).sum()

    precision = tp / (tp + fp) if (tp + fp) > 0 else np.nan
    recall = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else np.nan

    return pd.Series({
        "Precision": precision,
        "Recall": recall,
        "Accuracy": accuracy,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn
    })

## 3. Buy-and-Hold Backtest Function

In [3]:
def backtest_buy_and_hold(
    test_df: pd.DataFrame,
    price_col: str = "open",
    timestamp_col: str = None,
    transaction_cost: float = 0.00035
) -> pd.DataFrame:
    """
    Backtest a passive buy-and-hold strategy using open-to-open hourly returns.

    Strategy logic:
    - position = 1 at all times in the tradable test window
    - one-time entry transaction cost at the beginning
    - no further transaction costs since position never changes

    Parameters
    ----------
    test_df : pd.DataFrame
        Test-period data.
    price_col : str
        Column containing the open price.
    timestamp_col : str, optional
        Optional timestamp column for easier reading.
    transaction_cost : float
        Proportional cost per trade (0.00035 = 0.035%)

    Returns
    -------
    pd.DataFrame
        Backtest dataframe with raw returns, position, costs, net returns, equity curve.
    """
    df = test_df.copy().reset_index(drop=True)

    # Keep only necessary columns if timestamp exists
    cols_to_keep = [price_col]
    if timestamp_col is not None and timestamp_col in df.columns:
        cols_to_keep = [timestamp_col, price_col]
        df = df[cols_to_keep].copy()

    # Compute open-to-open simple return:
    # return from open_t to open_{t+1}
    df["raw_return"] = df[price_col].shift(-1) / df[price_col] - 1

    # The last row has no next-period return, so it is not tradable
    df = df.iloc[:-1].copy().reset_index(drop=True)

    # Constant long exposure
    df["position"] = 1

    # Position change:
    # first row is entry from 0 -> 1
    df["prev_position"] = df["position"].shift(1).fillna(0)
    df["position_change"] = (df["position"] - df["prev_position"]).abs()

    # Transaction cost applies whenever position changes
    df["transaction_cost"] = transaction_cost * df["position_change"]

    # Gross strategy return = position * raw_return
    df["gross_return"] = df["position"] * df["raw_return"]

    # Net return = gross return - transaction cost
    df["net_return"] = df["gross_return"] - df["transaction_cost"]

    # Equity curves
    df["gross_equity_curve"] = (1 + df["gross_return"]).cumprod()
    df["net_equity_curve"] = (1 + df["net_return"]).cumprod()

    return df

## 4. Run Buy-and-Hold Backtest

In [5]:
def run_buy_and_hold_backtest(
    test_df: pd.DataFrame,
    price_col: str = "open",
    timestamp_col: str = None,
    transaction_cost: float = 0.00035,
    periods_per_year: int = 8760,
    risk_free_rate: float = 0.0,
    include_directional_metrics: bool = True
):
    """
    Full pipeline:
    1. Run buy-and-hold backtest
    2. Compute performance summary
    3. Optionally compute directional metrics
    """
    backtest_df = backtest_buy_and_hold(
        test_df=test_df,
        price_col=price_col,
        timestamp_col=timestamp_col,
        transaction_cost=transaction_cost
    )

    performance_summary = summarize_performance(
        backtest_df=backtest_df,
        return_col="net_return",
        position_col="position",
        periods_per_year=periods_per_year,
        risk_free_rate=risk_free_rate
    )

    if include_directional_metrics:
        directional_summary = compute_directional_metrics(backtest_df)
        full_summary = pd.concat([performance_summary, directional_summary])
    else:
        full_summary = performance_summary

    return backtest_df, full_summary

In [7]:
btc_test = pd.read_csv("data/split_data/BTC_test.csv")
eth_test = pd.read_csv("data/split_data/ETH_test.csv")
xrp_test = pd.read_csv("data/split_data/XRP_test.csv")
sol_test = pd.read_csv("data/split_data/SOL_test.csv")

test_datasets = {
    "BTC": btc_test,
    "ETH": eth_test,
    "XRP": xrp_test,
    "SOL": sol_test
}

all_backtest_results = {}
all_summaries = []

for asset_name, test_df in test_datasets.items():
    backtest_results, summary = run_buy_and_hold_backtest(
        test_df=test_df,
        price_col="open",
        timestamp_col="timestamp",
        transaction_cost=0.00035,
        periods_per_year=8760,
        risk_free_rate=0.0,
        include_directional_metrics=True
    )

    all_backtest_results[asset_name] = backtest_results

    summary_df = summary.to_frame().T
    summary_df.insert(0, "Asset", asset_name)
    all_summaries.append(summary_df)

final_summary_table = pd.concat(all_summaries, ignore_index=True)

print("===== BUY-AND-HOLD SUMMARY FOR ALL ASSETS =====")
print(final_summary_table)

# save each backtest output
for asset_name, df in all_backtest_results.items():
    df.to_csv(f"result/buy_and_hold/{asset_name.lower()}_backtest.csv", index=False)

# save summary
final_summary_table.to_csv("result/buy_and_hold/summary_all_assets.csv", index=False)

===== BUY-AND-HOLD SUMMARY FOR ALL ASSETS =====
  Asset  Cumulative Return  Annualized Return  Annualized Volatility  \
0   BTC           0.419651           0.324142               0.457836   
1   ETH           0.198466           0.156108               0.711437   
2   XRP           2.047236           1.441894               0.950950   
3   SOL          -0.159752          -0.130175               0.858233   

   Annualized Sharpe  Maximum Drawdown  Turnover  Number of Trades  \
0           0.842262         -0.347615       1.0               1.0   
1           0.560312         -0.652824       1.0               1.0   
2           1.412405         -0.512126       1.0               1.0   
3           0.266212         -0.660623       1.0               1.0   

   Average Return per Trade  Precision  Recall  Accuracy      TP      FP   FN  \
0                  0.419651   0.507637     1.0  0.507637  5550.0  5383.0  0.0   
1                  0.198466   0.509101     1.0  0.509101  5566.0  5367.0  0.0 

While directional metrics such as precision and recall are reported for completeness, they are not economically meaningful for the buy-and-hold benchmark, as the strategy trivially predicts a positive return at every period. As a result, recall is mechanically equal to one and true negatives are zero.